# Balsara 1 shock tube test

This notebook runs a shock-tube test with AsterX, and visualizes the data saved in TSV and OpenPMD format. 


# Test setup
This is a generic shock-tube test with magnetic fields, where the shock propagates along x-direction. The initial conditions are the same as of the Balsara 1 shock tube problem. Here, the right and left states along x are initalized to different values as follows:

- Left state is initialized as ($\rho, v^x, v^y, v^z, P$) = $[1,0,0,1]$, and the right state as $[0.125,0,0,0.1]$. 

- For the magnetic field components, left state is ($B^x, B^y, B^z$) = $[0.5,1.0,0.0]$, and the right state $[0.5,-1.0,0.0]$.

- Here, we use the ideal gas EOS with $\gamma = 2$. 

- Initial data is set using the thorn ```AsterSeeds```.

- Grid domain is $[-0.5,+0.5]$ along x, with 200 grid-cells.

- `Linear extrapolation` is used as boundary conditions in all directions.

This gives the initial contact discontinuity at $x=0.0$.


# Reconstruction Method

This test will use the so-called MINMOD reconstruction method. It is a second-order reconstruction method designed for the stable evolution of sharp features such as shocks.

MINMOD applies a slope-limiter to reconstruct a function within a cell, it's implementation can be found in: `AsterX/ReconX/src/minmod.hxx`

## Reconstruction: the minmod scheme

Finite-volume schemes evolve **cell-averaged** values, but the Riemann solver at
each cell interface needs the values of the fields *at the face*. Reconstruction
is the step that rebuilds those face values from the surrounding cell averages.

A first-order scheme just uses the cell average itself (piecewise-constant). To
get **second-order** accuracy we instead assume the field varies *linearly*
inside each cell (piecewise-linear). The problem is that a naive slope
overshoots near shocks and produces spurious oscillations. A **slope limiter**
prevents this; here we use **minmod**, which yields a Total-Variation-Diminishing
(TVD) scheme.

### The minmod limiter

$$
\mathrm{minmod}(x,y) =
\begin{cases}
0, & \operatorname{sign}(x)\neq\operatorname{sign}(y)\\[2pt]
x, & |x| < |y|\\[2pt]
y, & \text{otherwise}
\end{cases}
\;=\;
\tfrac{1}{2}\big[\operatorname{sign}(x)+\operatorname{sign}(y)\big]\,
\min\!\big(|x|,|y|\big).
$$

In words: if the two candidate slopes disagree in sign (a local extremum), the
limited slope is set to **zero** — the cell falls back to first order, killing
the oscillation. Otherwise it returns the **smaller-magnitude** slope, i.e. the
least steep / most conservative one.

### Reconstructing the two states at a face

Consider the interface shared by two neighbouring cells. Label the four cells as

$$
\dots \;\;|\;\; q_{I^{--}} \;\;|\;\; q_{I^-} \;\;\mbox{\Large \textbf{|}}\;\; q_{I^+} \;\;|\;\; q_{I^{++}} \;\;|\;\;
\dots
$$

with the face sitting between $q_{I^-}$ and $q_{I^+}$. We form three
finite differences (slopes over one cell width):

$$
\Delta_m = q_{I^-} - q_{I^{--}}, \qquad
\Delta_c = q_{I^+} - q_{I^-},   \qquad
\Delta_p = q_{I^{++}} - q_{I^+}.
$$

The **left** state (value of cell $I^-$ extrapolated to its right face) and the
**right** state (value of cell $I^+$ extrapolated to its left face) are

$$
q^{L} = q_{I^-} + \tfrac{1}{2}\,\mathrm{minmod}(\Delta_c,\Delta_m),
\qquad
q^{R} = q_{I^+} - \tfrac{1}{2}\,\mathrm{minmod}(\Delta_p,\Delta_c).
$$

The factor $\tfrac{1}{2}$ is the half-cell distance from the cell centre to the
face (uniform grid). On a smooth profile minmod returns the genuine slope and
the reconstruction is second-order; near a discontinuity it clips the steeper
slope (or zeroes it at an extremum), keeping the scheme monotone.

The AsterX routine returns the pair $(q^{L}, q^{R})$ — in the code `{var_m, var_p}` —
which is then handed to the Riemann solver at that interface.

In [ ]:
"""
Demo of the minmod reconstruction used in ReconX, with a first-order
(piecewise-constant) reconstruction overlaid for comparison.

This cell reproduces ReconX::minmod / ReconX::minmod_reconstruct
in NumPy on a smooth Gaussian bump, resolved by only a handful of cells so the
limiter's per-cell slopes are easy to see.
"""

%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

# --------------------------------------------------------------------------
# NumPy port of ReconX::minmod / ReconX::minmod_reconstruct
# --------------------------------------------------------------------------
def minmod(x, y):
    """Scalar minmod limiter (matches ReconX::minmod)."""
    if np.sign(x) != np.sign(y):     # signs disagree -> local extremum
        return 0.0
    return x if abs(x) < abs(y) else y   # else the smaller-magnitude slope

def minmod_reconstruct(q_Imm, q_Im, q_Ip, q_Ipp):
    """Reconstruct the two states at the face between cells Im and Ip.
    Returns (qL, qR), matching ReconX::minmod_reconstruct -> {var_m, var_p}."""
    slope_p = q_Ipp - q_Ip
    slope_c = q_Ip  - q_Im
    slope_m = q_Im  - q_Imm
    qL = q_Im + 0.5 * minmod(slope_c, slope_m)   # var_m: Im on its right side
    qR = q_Ip - 0.5 * minmod(slope_p, slope_c)   # var_p: Ip on its left side
    return qL, qR

# --------------------------------------------------------------------------
# Test profile: a smooth Gaussian bump on [0.2, 0.4], few cells
# --------------------------------------------------------------------------
N  = 16
x  = np.linspace(0.2, 0.4, N)          # cell centres (uniform grid)
dx = x[1] - x[0]
q  = np.exp(-((x - 0.30) / 0.06)**2)   # smooth Gaussian bump

# --------------------------------------------------------------------------
# Reconstruct the face states
#   - second order: minmod limiter
#   - first  order: piecewise constant (qL = q_Im, qR = q_Ip)
# --------------------------------------------------------------------------
xf, qL, qR = [], [], []
for i in range(1, N - 2):              # need cells i-1 .. i+2
    L, R = minmod_reconstruct(q[i-1], q[i], q[i+1], q[i+2])
    xf.append(0.5 * (x[i] + x[i+1]))   # face position
    qL.append(L); qR.append(R)
xf, qL, qR = map(np.array, (xf, qL, qR))

# Per-cell limited slope (same minmod), used only to draw the
# piecewise-linear reconstruction inside each cell
slope = np.zeros_like(q)
for i in range(1, N - 1):
    slope[i] = minmod(q[i+1] - q[i], q[i] - q[i-1])

# --------------------------------------------------------------------------
# Plot
# --------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=[7.0, 4.0])

# cell averages
ax.plot(x, q, "o", color="0.4", ms=4, label="cell average $q_i$")

# first-order: piecewise-constant reconstruction inside each cell
for i in range(1, N - 1):
    ax.plot([x[i] - dx/2, x[i] + dx/2], [q[i], q[i]],
            color="seagreen", lw=1.0, ls="--", zorder=1,
            label="1st order (const)" if i == 1 else None)

# second-order: piecewise-linear reconstruction inside each cell
for i in range(1, N - 1):
    ax.plot([x[i] - dx/2, x[i] + dx/2],
            [q[i] - slope[i]/2, q[i] + slope[i]/2],
            color="0.7", lw=1.2, zorder=2,
            label="2nd order (minmod)" if i == 1 else None)

# reconstructed face states (second order)
ax.plot(xf, qL, "<", color="crimson",     ms=6, zorder=3, label=r"$q^{L}$ (left state)")
ax.plot(xf, qR, ">", color="midnightblue", ms=6, zorder=3, label=r"$q^{R}$ (right state)")

ax.set_title("Reconstruction: 1st order vs. minmod",
             fontsize=11, fontweight="bold")
ax.set_xlabel("x"); ax.set_ylabel("q")
ax.legend(fontsize=8, loc="upper right")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

# 1. Steps to perform the simulation

In [ ]:
# this allows you to use "cd" in cells to change directories instead of requiring "%cd"
%automagic on
from etsetup import *

First, let's first move to the Cactus folder:

In [ ]:
cd ~/Cactus

Now, let's create the parameter file to be used for this simulation:

In [ ]:
%%writefile ./par/Balsara1_shocktube_xdir_MINMOD.par

ActiveThorns = "
    CarpetX
    IOUtil
    ODESolvers
    TimerReport
    ADMBaseX
    HydroBaseX
    TmunuBaseX
    AsterMasks
    AsterSeeds
    AsterX
    EOSX
"
 
$nlevels = 1
$ncells = 200

Cactus::cctk_show_schedule = yes

Cactus::presync_mode = "mixed-error"

Cactus::terminate = "time"
Cactus::cctk_final_time = 0.40

ADMBaseX::initial_data            = "Cartesian Minkowski"
ADMBaseX::initial_lapse           = "one"
ADMBaseX::initial_shift           = "zero"
ADMBaseX::initial_dtlapse         = "none"
ADMBaseX::initial_dtshift         = "none"

CarpetX::verbose = no

CarpetX::xmin = -0.5
CarpetX::ymin = -0.5
CarpetX::zmin = -0.5

CarpetX::xmax = +0.5
CarpetX::ymax = +0.5
CarpetX::zmax = +0.5

CarpetX::boundary_x = "linear extrapolation"
CarpetX::boundary_y = "linear extrapolation"
CarpetX::boundary_z = "linear extrapolation"

CarpetX::boundary_upper_x = "linear extrapolation"
CarpetX::boundary_upper_y = "linear extrapolation"
CarpetX::boundary_upper_z = "linear extrapolation"

CarpetX::ncells_x = $ncells
CarpetX::ncells_y = 2
CarpetX::ncells_z = 2

CarpetX::max_num_levels = $nlevels
CarpetX::regrid_every = 100000
CarpetX::blocking_factor_x = 1
CarpetX::blocking_factor_y = 1
CarpetX::blocking_factor_z = 1
CarpetX::regrid_error_threshold = 0.01

CarpetX::prolongation_type = "ddf"
CarpetX::ghost_size = 3
CarpetX::dtfac = 0.25
 
AsterSeeds::test_type = "1DTest"
AsterSeeds::test_case = "Balsara1"
AsterSeeds::shock_dir = "x"

AsterX::debug_mode = "no"
AsterX::flux_type = "HLLE"
AsterX::update_tmunu = "no"

ReconX::reconstruction_method = "minmod"

Con2PrimFactory::c2p_prime = "Noble"
Con2PrimFactory::c2p_second = "Palenzuela"
Con2PrimFactory::c2p_tol = 1e-8
Con2PrimFactory::max_iter = 100
Con2PrimFactory::rho_abs_min = 1e-8
Con2PrimFactory::unit_test = "no"
Con2PrimFactory::B_lim = 1e8
Con2PrimFactory::vw_lim = 1e8
Con2PrimFactory::Ye_lenient = "yes"

EOSX::evolution_eos = "IdealGas"
EOSX::gl_gamma = 2.0
EOSX::poly_gamma = 2.0
EOSX::rho_max = 1e8
EOSX::eps_max = 1e8

ODESolvers::method = "RK4"

IO::out_dir = $parfile
IO::out_every = 10 

CarpetX::out_tsv_vars = "
    HydroBaseX::rho
    HydroBaseX::vel
    HydroBaseX::eps
    HydroBaseX::press
    HydroBaseX::Bvec
"

CarpetX::out_openpmd_vars = "
    HydroBaseX::rho
    HydroBaseX::vel
    HydroBaseX::eps
    HydroBaseX::press
    HydroBaseX::Bvec
"

TimerReport::out_every = 10
TimerReport::out_filename = "TimerReport"
TimerReport::output_all_timers_together = yes
TimerReport::output_all_timers_readable = yes
TimerReport::n_top_timers = 50

Then, submit the simulation using the following command:

In [ ]:
%%bash
rm -rf $HOME/simulations/Balsara1_shocktube_xdir_MINMOD
./simfactory/bin/sim create-submit Balsara1_shocktube_xdir_MINMOD \
  --parfile=./par/Balsara1_shocktube_xdir_MINMOD.par --procs=2 --num-threads=1 --walltime=00:20:00

The above command creates and runs the simulation ```Balsara1_shocktube_xdir_MINMOD```, using the configuration ```sim```. The data is saved in the directory ```./simulations/Balsara1_shocktube_xdir_MINMOD```.



In [ ]:
%%bash
# watch log output, following along as new output is produced
cd ~/Cactus
./simfactory/bin/sim show-output --follow Balsara1_shocktube_xdir_MINMOD

# 2. 1D Data Output -TSV Format

Norms and 1D data is saved using the TSV format which can be directly visualized using standard Python modules:

A small example toolkit for TSV data can be imported here:

  ## `carpetx_tsv_classes` — reading CarpetX `.tsv` output

  This helper module turns CarpetX `.tsv` output into NumPy arrays. It has two layers: **discovery**
  classes that scan an output folder and catalog what's available, and **loader** classes/functions
  that read a single file.

  ### Discovery — *what's in this output folder?*

  - **`OneDimTsv_OneOut(path)`** — scans `path` for 1D line-output files (`*.it*.tsv`) and parses 
  their names into variable / iteration / direction.
    - `get_vars()` → list of variables found
    - `get_iterations(var, direc)` → sorted array of iterations available for that variable and 
  direction (`'x'`, `'y'`, or `'z'`)
    - `get_path_list()` → all matching file paths

  - **`NormsTsv_OneOut(path)`** — scans `path` for **norm / time-series** files (`*.tsv` *without* an
  `.itNNNN` tag).
    - `get_vars()` → list of variables found
    - `get_path_list()` → all matching file paths

  ### Loaders — *read one file into arrays*
  
  - **`LoadTsv(path)`** — loads a single `.tsv` and exposes its columns by name.
    - `get(name)` → that column as a 1D array
    - `get_keys()` → available column names

  - **`LoadAllTimersTsv(path)`** — specialized loader for the `AllTimers.tsv` profiling file, which
  stores a different column set at `it=0` than at `it>0`. `get(name)` stitches the two together
  transparently. Also provides `get_clock()` and `get_clock_units()`. 

  ### Convenience functions — *discovery + loader in one step*

  - **`LoadNormsTsv(out, var)`** — given a `NormsTsv_OneOut` and a variable name, returns the right
  loader (`LoadAllTimersTsv` for `AllTimers`, otherwise `LoadTsv`).
  - **`LoadOneDimTsv(out, var, direction, it)`** — given a `OneDimTsv_OneOut`, returns a `LoadTsv` for
  that variable / direction / iteration.

  ### Typical usage

  ```python
  import carpetx_tsv_classes as ctx

  out  = ctx.OneDimTsv_OneOut("/path/to/simulation/output")
  print(out.get_vars())                         # which variables are available
  print(out.get_iterations("hydrobasex-rho", "x"))         # iterations along x for rho

  data = ctx.LoadOneDimTsv(out, "hydrobasex-rho", "x", 128) # load rho along x at iteration 128
  x    = data.get("x")
  rho  = data.get("rho")
  ```

  ### Norms / time-series example

  ```python
  import carpetx_tsv_classes as ctx

  norms = ctx.NormsTsv_OneOut("/path/to/simulation/output")
  print(norms.get_vars())              # which norm/time-series variables exist

  data = ctx.LoadNormsTsv(norms, "hydrobasex-rho")  # load the rho norm time series
  print(data.get_keys())               # available columns (e.g. iteration, time, ...)
  t   = data.get("time")
  rho = data.get("rho")

  # AllTimers profiling file works the same way (returns a LoadAllTimersTsv):
  timers = ctx.LoadNormsTsv(norms, "AllTimers")
  print(timers.get_clock(), timers.get_clock_units())
  ```

In [ ]:
import sys, os
sys.path.insert(0, os.path.expanduser("~/AsterX_Tutorial"))
import carpetx_tsv_classes as ctx

### Identify correct output directory:

In [ ]:
%%bash

ls /home/mchabanov/simulations/Balsara1_shocktube_xdir_MINMOD/output-0000/Balsara1_shocktube_xdir_MINMOD/hydrobasex-rho.it000000*

### Check available variables and iterations:

In [ ]:
out  = ctx.OneDimTsv_OneOut(os.path.expanduser("~/simulations/Balsara1_shocktube_xdir_MINMOD/output-0000/Balsara1_shocktube_xdir_MINMOD"))
print(out.get_vars())                                    # which variables are available
print(out.get_iterations("hydrobasex-rho", "x"))         # iterations along x for rho

### Load and plot variables:

In [ ]:
data = ctx.LoadOneDimTsv(out, "hydrobasex-rho", "x", 320) # load rho along x at iteration 320
x    = data.get("x")
rho  = data.get("rho")

# Set up the figure (no colorbar needed for a line plot)
fig    = plt.figure(figsize=[5.0, 2.75])
axplot = fig.add_axes([0.14, 0.16, 0.80, 0.74])

axplot.set_title("Rest-mass density", fontsize=10., fontweight="bold", color="midnightblue")
axplot.set_xlabel("x", fontsize=7.)
axplot.set_ylabel("rho", fontsize=7.)
axplot.tick_params(labelsize=7)
axplot.xaxis.set_major_locator(plt.MaxNLocator(5))
axplot.yaxis.set_major_locator(plt.MaxNLocator(5))

# sort by coordinate so the line is drawn left-to-right
order = np.argsort(x)
axplot.plot(x[order], rho[order], color="midnightblue", lw=1.2)
axplot.set_ylim(0.0, 1.1)

# Print the current iteration (axes-fraction coords, so it stays put)
axplot.text(0.70, 0.88, f"Iteration {320}", transform=axplot.transAxes,
            fontsize=8., fontweight="bold", color="black")

In [ ]:
plt.close(fig)

### Full movie script:

In [ ]:
"""
Animate 1D CarpetX .tsv line output as a movie (value vs. coordinate, one
frame per iteration), using the discovery/loader classes in
carpetx_tsv_classes.
"""

from celluloid import Camera
from IPython.display import display, HTML

# --------------------------------------------------------------------------
# Discovery object (point this at your simulation output folder)
# --------------------------------------------------------------------------
out = ctx.OneDimTsv_OneOut(os.path.expanduser("~/simulations/Balsara1_shocktube_xdir_MINMOD/output-0000/Balsara1_shocktube_xdir_MINMOD"))

# --- choose what to plot ---
var       = "hydrobasex-rho"   # a name from out.get_vars()
direction = "x"                # "x", "y", or "z"

iterations = out.get_iterations(var, direction)
print(f"{len(iterations)} iterations for {var} along {direction}")

# peek at one file so we know the exact column names
sample = ctx.LoadOneDimTsv(out, var, direction, iterations[0])
print("columns:", sample.get_keys())

# adjust these two if get_keys() shows different names:
coord_col = direction   # the coordinate column (usually "x"/"y"/"z")
value_col = "rho"       # the data column

# --------------------------------------------------------------------------
# Build the movie
# --------------------------------------------------------------------------
# Precompute a stable y-range so the axis doesn't jump between frames
ymin, ymax = np.inf, -np.inf
for it in iterations:
    v = ctx.LoadOneDimTsv(out, var, direction, it).get(value_col)
    ymin, ymax = min(ymin, v.min()), max(ymax, v.max())

# Set up the figure (no colorbar needed for a line plot)
fig    = plt.figure(figsize=[5.0, 2.75])
axplot = fig.add_axes([0.14, 0.16, 0.80, 0.74])

axplot.set_title("Rest-mass density", fontsize=10., fontweight="bold", color="midnightblue")
axplot.set_xlabel(direction, fontsize=7.)
axplot.set_ylabel(value_col, fontsize=7.)
axplot.tick_params(labelsize=7)
axplot.xaxis.set_major_locator(plt.MaxNLocator(5))
axplot.yaxis.set_major_locator(plt.MaxNLocator(5))

# Initialize the camera
camera = Camera(fig)

# Print frames
for it in iterations:
    data  = ctx.LoadOneDimTsv(out, var, direction, it)
    coord = data.get(coord_col)
    value = data.get(value_col)

    # sort by coordinate so the line is drawn left-to-right
    order = np.argsort(coord)
    axplot.plot(coord[order], value[order], color="midnightblue", lw=1.2)
    axplot.set_ylim(ymin, ymax)

    # Print the current iteration (axes-fraction coords, so it stays put)
    axplot.text(0.70, 0.88, f"Iteration {it}", transform=axplot.transAxes,
                fontsize=8., fontweight="bold", color="black")

    # Snapshot for the animation
    camera.snap()

animation = camera.animate(interval=120, blit=True)

# Inline playback in the notebook. display() makes it render even when this
# isn't the last line of the cell. plt.close() stops a static duplicate frame
# from also being shown.
plt.close(fig)
display(HTML(animation.to_jshtml()))

# --------------------------------------------------------------------------
# Optional: save to a file
# --------------------------------------------------------------------------
# animation.save("rho_1d.mp4", dpi=150)          # needs ffmpeg
# animation.save("rho_1d.gif", writer="pillow")  # no ffmpeg required

## 2. Steps to visualize 2D simulation data

The 2D data can be saved in both Silo format (which can be visualised, for instance, via VisIt) and in OpenPMD format. 

For further info on Silo, please visit: https://wci.llnl.gov/simulation/computer-codes/silo

For further info about OpenPMD, please visit:

- Official website:  https://www.openpmd.org
- GitHub repository: https://github.com/openPMD
- Documentation:     https://openpmd-api.readthedocs.io

Now, let's go back to the home directory:

In [ ]:
cd ~/

Import all the required modules:

In [ ]:
%pip install --user celluloid

In [ ]:
import numpy as np
from matplotlib import pyplot as plt
from celluloid import Camera
import openpmd_api as io

Set the Matplotlib backend to `notebook`, not `inline`, since we'll want to animate some figures and the latter is not compatible with that

In [ ]:
%matplotlib inline

Open a .bp file (ADIOS2 extension) as an OpenPMD **'series'**, which is a collection of **'iterations'**, each of which contains **'records'**, which are sets of either structured data --- **'meshes'** --- or unstructured data --- **'particles'**. 

AsterX only outputs mesh data. Each record has one or more **'components'**: for example, a record representing a scalar field only has one component, while a record representing a vector field has three.

In [ ]:
home = os.environ["HOME"]
series = io.Series(os.path.join(home, "simulations",
                                "Balsara1_shocktube_xdir_MINMOD",
                                "output-0000","Balsara1_shocktube_xdir_MINMOD",
                                "Balsara1_shocktube_xdir_MINMOD.it%08T.bp5"), io.Access.read_only)

All iterations in our series have the same structure, i.e., they contain
the same records, since they all represent the same output, just at
different times. 

Here we define an empty Python nested dictionary whose
structure, once full, will be:

Iteration 0:
- Record 1:
  - Component 1: 3D data array
  - Component 2: 3D data array
  - Component 3: 3D data array
- Record 2:
  - Component 1: 3D data array
  - Component 2: 3D data array
  - Component 3: 3D data array
  
 [...]

Iteration 1:
- Record 1:
  - Component 1: 3D data array
  - Component 2: 3D data array
  - Component 3: 3D data array
- Record 2:
  - Component 1: 3D data array
  - Component 2: 3D data array
  - Component 3: 3D data array
  
 [...]

[...]

In [ ]:
iter_rec_comp_dict = {}

Print info, register data chunks and fill the above dictionary:

In [ ]:
for index in series.iterations:
    iteration = str(index)

    print("\nIteration " + iteration + ":")
    print("==============")

    # Allocate an empty dictionary associated to this iteration
    iter_rec_comp_dict[iteration] = {}

    i = series.iterations[index]

    for key in i.meshes:
        print("Components of record \"" + key + "\":")

        # Allocate an empty dictionary associated to this record. Notice that
        # 'record' is an OpenPMD mesh object, so it's better to use 'key'
        # instead of 'record' as a key in the dictionary ('record' could also be
        # used, but it makes accessing the key clumsy).
        record = i.meshes[key]
        iter_rec_comp_dict[iteration][key] = {}

        # Load each component of each record as a 'data chunk', i.e., an
        # allocated, but STILL INVALID, NumPy array. Later we will flush all
        # chunks (i.e., basically, fill the NumPy arrays) at once: this leads
        # to better I/O performance compared to flushing a large number of
        # small chunks. That's why we bothered creating the nested dictionary:
        # in this way, we can access the valid NumPy arrays for plotting
        # without having to flush each single chunk.
        # *IMPORTANT*: DO NOT access data chunks until flushing has happened!
        for component in record:
            print("    > " + component)  # 'component' is a string
            iter_rec_comp_dict[iteration][key][component] = record[component].load_chunk()  # *INVALID* 3D NumPy array

        print("")

Flush all registered data chunks, which are now **VALID** 3D NumPy arrays:

In [ ]:
series.flush()

Visualize a 2D movie of the mass density on the xz plane:

In [ ]:
# Select the desired record and component to plot
record    = "hydrobasex_rho_patch00_lev00"  # "hydrobasex_eps_patch00_lev00", "hydrobasex_press_patch00_lev00", "hydrobasex_vel_patch00_lev00"
component = "hydrobasex_rho"  # "hydrobasex_eps", "hydrobasex_press", "hydrobasex_rho", "hydrobasex_velx", "hydrobasex_vely", "hydrobasex_velz"

# Set up the axes for the plot and the colorbar
fig    = plt.figure(figsize = [5.0, 2.75])
axplot = fig.add_axes([0.12, 0.14, 0.75, 0.75])
axclb  = fig.add_axes([0.88, 0.14, 0.02, 0.75])

# Set title and labels
axplot.set_title("Rest-mass density", fontsize = 10., fontweight = "bold", color = "midnightblue")
axplot.set_xlabel("x", fontsize = 7.)
axplot.set_ylabel("z", fontsize = 7.)
axplot.tick_params(labelsize=7)
axplot.xaxis.set_major_locator(plt.MaxNLocator(5))
axplot.yaxis.set_major_locator(plt.MaxNLocator(5))


# Initialize the camera
camera = Camera(fig)

# Print frames
for iteration in iter_rec_comp_dict:
    # Retrieve the 3D array containing the data
    array3D = iter_rec_comp_dict[iteration][record][component]
    
    # Plot on the (x, z) plane at the half-way value of y
    # Notice that the 3D array is stored in (z, y, x) order
    y_index = int(array3D.shape[1]/2)
    x0     = np.linspace(-0.5, 0.5, array3D.shape[2])
    z0     = np.linspace(-0.04, 0.04, array3D.shape[0])
    image   = axplot.pcolormesh(x0, z0, array3D[:, y_index, :],
                                cmap = "magma", vmin = 0.0, vmax = 1.0)
    axplot.set_ylim(ymin=-0.005, ymax=0.005)
    # Set up the colorbar
    axclb.tick_params(labelsize=7.0)
    fig.colorbar(image, cax = axclb, extend = "neither")
    
    # Print the current iteration
    axplot.text(0.18, 0.42, "Iteration " + iteration,
             fontsize = 8., fontweight = "bold", color = "white")

    # Take a snapshot of the figure at this iteration (needed later for the animation)
    camera.snap()

    plt.close(fig)

In [ ]:
from IPython.display import HTML
animation = camera.animate()
HTML(animation.to_html5_video())

You may (or may not) delete the simulation directory via the following:


In [ ]:
#%cd ~/
#%rm -r ./simulations/Balsara1_shocktube_xdir_MINMOD

# We need the data from this simulation for our next notebook!
# Delete only once we are done!